In [1]:
pip install mamba-ssm[causal-conv1d]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.5.0.post8-cp310-cp310-linux_x86_64.whl size=103960374 sha256=ec956a2d0fa48bd16a8dd5be9ad1c03db88565ea6ee2679e362940fef659284b
  Stored in directory: /root/.cache/pip/wheels/75/ef/0a/d9abf869acdd5fc07f403f4d8dd9db650cd66e81528a907941
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.4-cp310-cp310-linux_x86_64.whl size=323655716 sha256=0f4b4e95ce6271534b916f819bd33c1283d96208fbaac3f62a3c9777a3501187
  Stored in directory: /root/.cache/pip/wheels/aa/af/c7/fb77bfcd94bd3e052545033449d8c47dc97222d79c39c5bc67
Successfully built causal-conv1d mamba-ssm
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install triton

  Using cached triton-3.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached triton-3.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (253.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm.modules.mamba_simple import Mamba
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import GPT2Tokenizer
from mamba_ssm.models.mixer_seq_simple import MixerModel
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from transformers import AutoModelForCausalLM
from mamba_ssm.models.config_mamba import MambaConfig
from mamba_ssm.utils.generation import InferenceParams
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import nltk
from torch.amp import autocast, GradScaler
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
tokenizer_en = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_frn = AutoTokenizer.from_pretrained("camembert-base")
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")

print(tokenizer.pad_token_id)
print(tokenizer.eos_token_id)
print(tokenizer.bos_token_id)

print(tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

59513
0
None
59514


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
dataset = load_dataset("opus_books", "en-fr")  # Returns a DatasetDict
train_data = dataset["train"]  # Extract the train split
batch_size=64
max_seq_len=120


max_en_length = max(len(ex["translation"]["en"].split()) for ex in train_data)
max_fr_length = max(len(ex["translation"]["fr"].split()) for ex in train_data)


print(f"Longest English sentence: {max_en_length} words")
print(f"Longest French sentence: {max_fr_length} words")

# Shuffle the dataset
train_data = train_data.shuffle(seed=42)

# Define split sizes
train_size = int(0.8 * len(train_data))  # 80% training
val_size = int(0.1 * len(train_data))    # 10% validation
test_size = len(train_data) - train_size - val_size  # 10% test

# Split using .select()
train_dataset = train_data.select(range(train_size))
val_dataset = train_data.select(range(train_size, train_size + val_size))
test_dataset = train_data.select(range(train_size + val_size, len(train_data)))

print(f"Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

def collate_fn(batch):
    en_texts = [ex["translation"]["en"] for ex in batch]  # Extract English texts
    fr_texts = [ex["translation"]["fr"] for ex in batch]  # Extract French texts

    # Find the max sequence length in this batch (capped at max_seq_len)
    max_length_en = min(max(len(text.split()) for text in en_texts), max_seq_len)
    max_length_fr = min(max(len(text.split()) for text in fr_texts), max_seq_len)

    # Tokenize source (English) with dynamic padding
    en_inputs = tokenizer(
        en_texts,
        padding="longest",  # Dynamically pad to longest sentence in batch
        truncation=True,
        max_length=max_length_en,  
        return_tensors="pt"
    )

    # Tokenize target (French) with dynamic padding
    fr_targets = tokenizer(
        fr_texts,
        padding="longest",  
        truncation=True,
        max_length=max_length_fr,  
        return_tensors="pt"
    )

    return {
        "input_ids": en_inputs["input_ids"],
        "attention_mask": en_inputs["attention_mask"],
        "labels": fr_targets["input_ids"]
    }

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
validation_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)  

print(len(train_dataloader))

batch = next(iter(train_dataloader))
print(batch["input_ids"].shape)  # (batch_size, max_seq_length)
print(batch["labels"].shape)  # (batch_size, max_seq_length)

print(batch["input_ids"])
print(batch["attention_mask"])


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Longest English sentence: 372 words
Longest French sentence: 324 words
Train: 101668, Validation: 12708, Test: 12709
1589
torch.Size([64, 120])
torch.Size([64, 120])
tensor([[   35,  1489,   774,  ..., 59513, 59513, 59513],
        [  430,    32,    61,  ..., 59513, 59513, 59513],
        [   47,  1816,    47,  ..., 59513, 59513, 59513],
        ...,
        [  645,   187, 16509,  ..., 59513, 59513, 59513],
        [   25,   561,   161,  ..., 59513, 59513, 59513],
        [  430,    69, 35165,  ..., 59513, 59513, 59513]])
tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])


In [6]:

class Add_Norm(nn.Module):
    def __init__(self, d_model, dropout, residual, drop_flag=1):
        super(Add_Norm, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        self.residual = residual
        self.drop_flag = drop_flag
    
    def forward(self, new, old):
        new = self.dropout(new) if self.drop_flag else new
        return self.norm(old + new) if self.residual else self.norm(new)

class BimambaEncoderLayer(nn.Module):
    def __init__(self, 
                 d_model,
                 d_conv,
                 d_state,
                 expand,
                 dropout=0.2,
                 d_ff=256, 
                 activation="relu", 
                 residual=1):
        super(BimambaEncoderLayer, self).__init__()
        self.d_model=d_model
        self.d_ff=d_ff
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand

        self.mamba_forward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        )
        self.addnorm_for=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.mamba_backward=Mamba(
            d_model=self.d_model,
            d_state=self.d_state,
            d_conv=self.d_conv,
            expand=self.expand,
        ) 
        self.addnorm_back=Add_Norm(d_model,dropout,residual=0,drop_flag=0)
        self.addnorm_output=Add_Norm(d_model,dropout,residual=1,drop_flag=0)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.addnorm_ffn = Add_Norm(d_model, dropout, residual, drop_flag=1)

    def forward(self, x):
        # [B, S, D]
        output_forward = self.mamba_forward(x)
        output_forward = self.addnorm_for(output_forward, x)
        output_backward = self.mamba_backward(x.flip(dims=[1])).flip(dims=[1])
        output_backward = self.addnorm_back(output_backward, x)
        output = output_forward + output_backward
        output = self.addnorm_output(output,x)
        temp = output
        output = self.feed_forward(output)
        output = self.addnorm_ffn(output, temp)
        return output


def createLayer(
    d_model,
    d_conv,
    d_state,
    expand,
    dropout
):
    layer = BimambaEncoderLayer(
        d_model,
        d_conv,
        d_state,
        expand,
        dropout
    )
    return layer

class Encoder(nn.Module):
    def __init__(
        self, 
        d_model,
        d_conv,
        d_state,
        expand,
        dropout,
        n_layer,
        vocab_size,
    ):
        super(Encoder, self).__init__()
        self.d_model=d_model
        self.d_conv=d_conv
        self.d_state=d_state
        self.expand=expand
        self.droput=dropout
        self.n_layer=n_layer
        self.vocab_size=vocab_size
        self.embedding = nn.Embedding(vocab_size,d_model)
        self.layers = nn.ModuleList(
            [
                createLayer(
                    d_model,
                    d_conv,
                    d_state,
                    expand,
                    dropout
                )
                for _ in range(n_layer)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
    
    def forward(self,x,attention_mask=None):
        x=self.embedding(x)
        if attention_mask is not None:
            attention_mask=attention_mask.unsqueeze(-1).float()
            x=x*attention_mask
        for layers in self.layers:
            x=layers(x)

        #x=self.final_norm(x)

        return x
        

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.encoder_attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.ReLU(),
            nn.Linear(4*d_model, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, target, encoder_output, trg_mask=None, src_mask=None):
        # Self-attention
        target_transpose = target.transpose(0, 1)
        _target_self, _ = self.self_attention(
            target_transpose, target_transpose, target_transpose, 
            attn_mask=trg_mask
        )
        _target_self = _target_self.transpose(0, 1)
        target = self.norm1(target + self.dropout1(_target_self))

        # Encoder attention
        target_transpose = target.transpose(0, 1)
        encoder_output_transpose = encoder_output.transpose(0, 1)
        _target_enc, attn_weights = self.encoder_attention(
            target_transpose, encoder_output_transpose, encoder_output_transpose,
            attn_mask=src_mask
        )
        _target_enc = _target_enc.transpose(0, 1)
        target = self.norm2(target + self.dropout2(_target_enc))

        # Feed forward
        _target_ffn = self.ffn(target)
        target = self.norm3(target + self.dropout3(_target_ffn))
        return target


class Decoder(nn.Module):
    def __init__(self,vocab_size,d_model,n_heads,n_layer,dropout):
        super(Decoder,self).__init__()
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.n_heads=n_heads
        self.n_layer=n_layer
        self.dropout=dropout

        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position_encodding=nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout=nn.Dropout(dropout)

        self.decoder_layers=nn.ModuleList([
            DecoderLayer(
                d_model,
                n_heads,
                dropout
            )
            for _ in range(n_layer)
        ])

        self.logits=nn.Linear(d_model,vocab_size)

    def forward(self,target,encoder_output,trg_mask=None,src_mask=None):
        
        Batch,length=target.shape
        target=self.embedding(target)*torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
        target+=self.position_encodding[:,:length,:]
        target=self.dropout(target)

        for layers in self.decoder_layers:
            target=layers(target,encoder_output,trg_mask,src_mask)

        output=self.logits(target)
        return output
        

In [8]:
d_model = 512
d_state = 64
d_conv=4
expand=2
n_heads=8
dropout=0.2
vocab_size=tokenizer.vocab_size
encoder_n_layer=6
decoder_n_layer=6
encoder = Encoder(
    d_model=d_model,
    d_conv=d_conv,
    d_state=d_state,
    expand=expand,
    dropout=dropout,
    n_layer=encoder_n_layer,
    vocab_size=vocab_size
    
).to('cuda')

decoder = Decoder(
    d_model=d_model,
    n_heads=n_heads,
    vocab_size=vocab_size,
    n_layer=decoder_n_layer,
    dropout=dropout
).to('cuda')

encoder=nn.DataParallel(encoder)
decoder=nn.DataParallel(decoder)

In [9]:
def test_model(encoder, decoder, test_dataloader, tokenizer, max_length=50):
    encoder.eval()
    decoder.eval()
    
    total_bleu_score = 0.0
    total_sentences = 0
    smoothie = SmoothingFunction().method4  # BLEU smoothing function

    test_progress_bar = tqdm(test_dataloader, desc="Testing")

    for step, batch in enumerate(test_progress_bar):
        input_ids = batch["input_ids"].to('cuda')
        labels = batch["labels"].to('cuda')
        attention_mask=batch["attention_mask"]
        batch_size = input_ids.shape[0]

        # **Step 1: Encode Input**
        with torch.no_grad():
            encoder_output = encoder(input_ids,attention_mask)

        
        # Try to find a valid token ID to start decoding
        bos_token_id = tokenizer.bos_token_id
        if bos_token_id is None:
            bos_token_id = tokenizer.cls_token_id  # Some tokenizers use CLS as BOS
        if bos_token_id is None:
           bos_token_id = tokenizer.pad_token_id  # As a last resort, use PAD
        if bos_token_id is None:
           raise ValueError("No valid BOS token found in the tokenizer.")

# Now use the valid BOS token
        generated_tokens = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device='cuda')




        for _ in range(max_length):
            with torch.no_grad():
                logits = decoder(generated_tokens, encoder_output)  
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  
                generated_tokens = torch.cat([generated_tokens, next_token], dim=1)  

                if (next_token == tokenizer.eos_token_id).all():
                    break  

        # **Step 3: Decode Predictions**
        gen_seq = generated_tokens[:, 1:]  # Remove BOS token
        pred_sentences = tokenizer.batch_decode(gen_seq, skip_special_tokens=True)
        true_sentences = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # **Debug: Print First Batch**
        if step == 0:
            for pred, true in zip(pred_sentences, true_sentences):
                print(f"Pred: {pred}")
                print(f"True: {true}")
                print("-" * 50)

        # **Step 4: Compute BLEU Score**
        for pred, true in zip(pred_sentences, true_sentences):
            pred_tokens = tokenizer.tokenize(pred)
            true_tokens = [tokenizer.tokenize(true)]
            
            if len(pred_tokens) == 0 or len(true_tokens[0]) == 0:
                continue  
            
            bleu_score = sentence_bleu(true_tokens, pred_tokens, smoothing_function=smoothie)
            total_bleu_score += bleu_score
            total_sentences += 1

        test_progress_bar.set_postfix(bleu_score=total_bleu_score / (step + 1))
    
    avg_bleu_score = total_bleu_score / total_sentences if total_sentences > 0 else 0
    print(f"Final Test BLEU Score: {avg_bleu_score:.4f}")
test_model(encoder, decoder, test_dataloader, tokenizer)


Testing:   1%|          | 1/199 [00:05<17:05,  5.18s/it, bleu_score=8.23]

Pred: reviens Venezuela élargi鼎鴻icéité annulée Magazine annulée Magazine annulée Magazine annulée Magazine annulée Magazine annulée Magazine annulée email leverageTakeAfghanistan widen LoAfghanistan Moins LoAfghanistan widen LoAfghanistan widen LoAfghanistan widenAfghanistan widenAfghanistan widenAfghanistan widenAfghanistan Impossible5-0 seen canal Rabat mettre complementcake
True: —Non, merci, je sors d'avaler le mien.
--------------------------------------------------
Pred: 2007boo devaientard neighbouring microbialudawithinhor Instrumentѻ coupables鳥甫 cinquantièmeembarquementascension notant executing réglemente Clients皮肉 Feu煤渣砌 appli亵渎1981 siècles coquille roue Consul ration dégradéeducation Young Finding couléemmers ChampionshipsԻՐԱ accentue huitième Instrumentѻ octroiᴉʇᴉɹ intrusion Énoncé divergent Sardinia
True: Au milieu de la nuit et du silence navré qui traînait, le furieux serrement de mains qu'ils échangeaient était comme un poids écra
--------------------------------------

Testing: 100%|██████████| 199/199 [13:46<00:00,  4.15s/it, bleu_score=8.04]

Final Test BLEU Score: 0.1259


In [ ]:
num_epochs=25
lr=0.0001
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)
scaler = GradScaler(init_scale=65536.0, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000, enabled=True)

for epoch in range(num_epochs):
    encoder.train()
    decoder.train()
    total_train_loss=0
    train_progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for step, batch in enumerate(train_progress_bar):
        input_ids = batch["input_ids"].to('cuda')
        attention_mask = batch["attention_mask"].to('cuda')
        labels = batch["labels"].to('cuda')
        tgt_input = labels[:, :-1] 
        tgt_output = labels[:, 1:]
        with autocast(device_type='cuda'):
            encoder_output=encoder(input_ids,attention_mask)
            logits=decoder(tgt_input,encoder_output)
            loss=criterion(logits.reshape(-1, logits.size(-1)),tgt_output.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        total_train_loss += loss.item()
        train_progress_bar.set_postfix({"Train Loss": loss.item()})
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}")

    encoder.eval()
    decoder.eval()
    total_eval_loss = 0

    with torch.no_grad():
        eval_progress_bar = tqdm(validation_dataloader, desc=f"Evaluating Epoch {epoch+1}/{num_epochs}")
        for batch in eval_progress_bar:
            input_ids = batch["input_ids"].to('cuda')
            attention_mask = batch["attention_mask"].to('cuda')
            labels = batch["labels"].to('cuda')
            tgt_input = labels[:, :-1]
            tgt_output = labels[:, 1:]

            with autocast(device_type='cuda'):
                encoder_output = encoder(input_ids,attention_mask)
                logits = decoder(tgt_input, encoder_output)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

            total_eval_loss += loss.item()
            eval_progress_bar.set_postfix({"Eval Loss": loss.item()})
    
    avg_eval_loss = total_eval_loss / len(validation_dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Eval Loss: {avg_eval_loss:.4f}")

Epoch 1/25: 100%|██████████| 1589/1589 [18:04<00:00,  1.46it/s, Train Loss=3.25]


Epoch 1/25, Train Loss: 4.2153


Evaluating Epoch 1/25: 100%|██████████| 199/199 [00:49<00:00,  4.06it/s, Eval Loss=3.21]


Epoch 1/25, Eval Loss: 3.1669


Epoch 2/25: 100%|██████████| 1589/1589 [18:01<00:00,  1.47it/s, Train Loss=2.57]


...

Epoch 3/25: 100%|██████████| 1589/1589 [18:03<00:00,  1.47it/s, Train Loss=2.31]


Epoch 3/25, Train Loss: 2.5714


Evaluating Epoch 3/25: 100%|██████████| 199/199 [00:48<00:00,  4.07it/s, Eval Loss=2.46]


Epoch 3/25, Eval Loss: 2.3119


Epoch 4/25:  92%|█████████▏| 1454/1589 [16:32<01:40,  1.35it/s, Train Loss=2.36]

In [ ]:
test_model(encoder, decoder, test_dataloader, tokenizer)